# Print fine-tuning paths

In [ ]:
import os
import glob

file_dir = '/data/yutong/synthetic_data/'
exp_pattern = 'sst2-validation-phi-nreal50-steps30-nsyn10-admm'
json_pattern = 'synthetic_data_clean_remove_cls_phi_sst2_positive_negative_instrFalse_fsTrue_top*.jsonl'

# Drill: file_dir / <run_dir>* / SST2* / <top*.jsonl>
training_dir = glob.glob(os.path.join(file_dir, f'{exp_pattern}*', 'SST2*', json_pattern))

print(len(training_dir))


In [4]:
for run in training_dir:
    print(' ' * 4 + run)

# Collect fine-tuning results

In [ ]:
import os
import glob
import json
from collections import defaultdict

import pandas as pd

# Point at all launch dirs to merge — list expands as new launches happen
list_exp_path = sorted(glob.glob('/home/yutong/online_admm/GRADMM/addax/synthetic_data_FT/*/result'))
print(f"Found {len(list_exp_path)} launch dirs:")
for p in list_exp_path:
    print(f"  {p}")

list_param = ["syn_data_path", "per_device_train_batch_size", "learning_rate", "max_steps", "num_train", "model_name", "task_name", "train_set_seed"]
list_metric = ["best_valid_acc", "best_valid_step", "best_valid_per_class_acc", "best_test_metric", "best_test_step", "best_test_per_class_acc"]

df_data = defaultdict(list)
n_skipped = 0
for exp_path in list_exp_path:
    for output_path in glob.glob(os.path.join(exp_path, '*')):
        results_file = os.path.join(output_path, "output/main_results.json")
        if not os.path.exists(results_file):
            n_skipped += 1
            continue
        main_results = json.load(open(results_file))
        for p in list_param:
            df_data[p].append(main_results["args"].get(p))
        for m in list_metric:
            df_data[m].append(main_results.get(m))

print(f"\nLoaded {len(df_data['learning_rate'])} runs (skipped {n_skipped} unfinished)")
df = pd.DataFrame(df_data)

import re
def short_tag(p):
    dp = re.search(r'dp_eps([0-9.]+)', p)
    rho = re.search(r'-rho([0-9.]+)-inner', p)
    alpha = re.search(r'_score_alpha([0-9.]+)_', p)
    return f"dp={dp.group(1) if dp else 'none'} rho={rho.group(1) if rho else '?'} alpha={alpha.group(1) if alpha else '?'}"

df['config'] = df['syn_data_path'].apply(short_tag)
df['key'] = df['config'] + ' lr=' + df['learning_rate'].astype(str)

print("\n=== Top single-seed runs (max over individual jobs) ===")
print(df.sort_values('best_test_metric', ascending=False)[['config', 'learning_rate', 'train_set_seed', 'best_test_metric']].head(10).to_string(index=False))

# Aggregate over seeds: mean ± std per (config, lr)
print("\n=== Multi-seed aggregated (mean ± std over seeds) ===")
agg = df.groupby(['config', 'learning_rate']).agg(
    mean_acc=('best_test_metric', 'mean'),
    std_acc=('best_test_metric', 'std'),
    n_seeds=('best_test_metric', 'count'),
).reset_index()
agg = agg.sort_values('mean_acc', ascending=False)
print(agg.head(15).to_string(index=False))

# Paper-style best: max-mean per config (across LRs)
print("\n=== Best mean-over-seeds per syn config (max across LRs) ===")
best_per_config = agg.groupby('config').agg(
    best_mean=('mean_acc', 'max'),
).sort_values('best_mean', ascending=False)
print(best_per_config.to_string())

print(f"\n=== Headline numbers (best mean over seeds) ===")
agg['has_dp'] = agg['config'].str.contains('dp=0.05')
for has_dp, grp in agg.groupby('has_dp'):
    label = 'DP eps=0.05' if has_dp else 'non-DP'
    best_row = grp.loc[grp['mean_acc'].idxmax()]
    paper = 88.0 if has_dp else 89.7
    print(f"  {label}: {best_row['mean_acc']*100:.1f} ± {best_row['std_acc']*100:.1f} (n={int(best_row['n_seeds'])} seeds) "
          f"@ {best_row['config']} lr={best_row['learning_rate']:.0e}   vs paper {paper}")
